# Car Mileage Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `mpg`

## 2 · Project Overview

We predict **car fuel efficiency** (MPG) based on engine specs (cylinders, displacement, horsepower), vehicle weight, model year, origin, fuel type, transmission, and drivetrain.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a car's engine specifications, weight, year, and configuration (fuel type, transmission, drivetrain), predict its fuel efficiency in miles per gallon.

## 5 · Why This Project Matters

- **Fuel economy** directly impacts vehicle costs and environmental emissions.
- This is a classic ML regression benchmark (auto-mpg).
- Understanding which factors drive MPG enables smarter car purchases.
- Demonstrates regression with strong physics-based relationships.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 6,000 |
| **Features** | cylinders, displacement_cc, horsepower, weight_lbs, acceleration_0_60, model_year, origin, fuel_type, transmission, drivetrain |
| **Target** | mpg (continuous) |
| **Range** | ~8 – 65 MPG |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "mpg"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 6,000 car records with engine and vehicle specifications.

In [ ]:
np.random.seed(SEED)
n = 6000
cylinders = np.random.choice([3, 4, 5, 6, 8, 10, 12], n, p=[0.05, 0.35, 0.05, 0.25, 0.2, 0.05, 0.05])
displacement_cc = np.round(cylinders * np.random.uniform(200, 600, n), 0).astype(int)
horsepower = np.round(25 * cylinders + np.random.normal(0, 20, n), 0).clip(50, 600).astype(int)
weight_lbs = np.round(1200 + 200 * cylinders + 0.3 * displacement_cc + np.random.normal(0, 200, n), 0).astype(int)
acceleration_0_60 = np.round(18 - 0.8 * cylinders + np.random.normal(0, 1.5, n), 1).clip(3, 20)
model_year = np.random.randint(2005, 2025, n)
origin = np.random.choice(["USA", "Europe", "Japan", "Korea"], n, p=[0.3, 0.25, 0.25, 0.2])
origin_eff = {"USA": 0.9, "Europe": 1.0, "Japan": 1.1, "Korea": 1.05}
orig_val = np.array([origin_eff[o] for o in origin])

fuel_type = np.random.choice(["Gasoline", "Diesel", "Hybrid"], n, p=[0.6, 0.2, 0.2])
fuel_eff = {"Gasoline": 1.0, "Diesel": 1.15, "Hybrid": 1.4}
fuel_val = np.array([fuel_eff[f] for f in fuel_type])

transmission = np.random.choice(["Manual", "Automatic", "CVT"], n, p=[0.25, 0.5, 0.25])
trans_eff = {"Manual": 1.0, "Automatic": 0.95, "CVT": 1.05}
trans_val = np.array([trans_eff[t] for t in transmission])

drivetrain = np.random.choice(["FWD", "RWD", "AWD"], n, p=[0.4, 0.3, 0.3])
drive_eff = {"FWD": 1.05, "RWD": 1.0, "AWD": 0.9}
drive_val = np.array([drive_eff[d] for d in drivetrain])

# MPG model (fuel economy inversely related to weight/power)
base_mpg = 55 - 0.008 * weight_lbs - 0.02 * horsepower + 0.3 * (model_year - 2005)
mpg = base_mpg * orig_val * fuel_val * trans_val * drive_val
mpg = np.round(mpg + np.random.normal(0, 2, n), 1).clip(8, 65)

df = pd.DataFrame({
    "cylinders": cylinders, "displacement_cc": displacement_cc,
    "horsepower": horsepower, "weight_lbs": weight_lbs,
    "acceleration_0_60": acceleration_0_60, "model_year": model_year,
    "origin": origin, "fuel_type": fuel_type,
    "transmission": transmission, "drivetrain": drivetrain,
    "mpg": mpg
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['mpg'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["weight_lbs"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Weight (lbs)"); axes[0][0].set_ylabel("MPG")
axes[0][0].set_title("Weight vs MPG")

axes[0][1].scatter(df["horsepower"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Horsepower"); axes[0][1].set_ylabel("MPG")
axes[0][1].set_title("Horsepower vs MPG")

axes[0][2].scatter(df["displacement_cc"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("Displacement (cc)"); axes[0][2].set_ylabel("MPG")
axes[0][2].set_title("Displacement vs MPG")

df.boxplot(column=TARGET, by="fuel_type", ax=axes[1][0])
axes[1][0].set_title("MPG by Fuel Type")

df.boxplot(column=TARGET, by="origin", ax=axes[1][1])
axes[1][1].set_title("MPG by Origin")

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `mpg`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Miles Per Gallon")

df.boxplot(column=TARGET, by="cylinders", ax=axes[1])
axes[1].set_title("MPG by Cylinder Count")

plt.tight_layout()
plt.show()
print(f"Range: {df[TARGET].min():.1f} to {df[TARGET].max():.1f} MPG")
print(f"Mean: {df[TARGET].mean():.1f}, Median: {df[TARGET].median():.1f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `origin`, `fuel_type`, `transmission`, `drivetrain` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Very low MPG (large trucks/sports cars) and high MPG (hybrids) are genuine.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["power_to_weight"] = X_train["horsepower"] / (X_train["weight_lbs"] + 1)
X_test["power_to_weight"] = X_test["horsepower"] / (X_test["weight_lbs"] + 1)

X_train["displacement_per_cyl"] = X_train["displacement_cc"] / (X_train["cylinders"] + 0.1)
X_test["displacement_per_cyl"] = X_test["displacement_cc"] / (X_test["cylinders"] + 0.1)

X_train["car_age"] = 2025 - X_train["model_year"]
X_test["car_age"] = 2025 - X_test["model_year"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Weight** is the strongest predictor (heavier = lower MPG).
- **Horsepower** has a strong negative correlation with MPG.
- **Fuel type** — Hybrid: 1.4× efficiency, Diesel: 1.15×.
- **Model year** shows improving efficiency over time (~0.3 MPG/year).
- **Origin** — Japanese cars slightly more efficient.
- **AWD** reduces MPG by ~10% vs FWD.

**Business takeaway:** Weight reduction is the most effective path to fuel efficiency. Hybrid/diesel drivetrains provide the largest technology-driven improvements.

## 26 · Limitations

1. Synthetic data based on auto-mpg patterns.
2. No aerodynamic drag coefficient.
3. Missing tire rolling resistance.
4. No city vs highway MPG split.
5. Simplified fuel type efficiency factors.

## 27 · How to Improve This Project

1. Use real EPA fuel economy data (fueleconomy.gov).
2. Add body style, aerodynamics, tire data.
3. Split city/highway/combined MPG.
4. Include electric and plug-in hybrid vehicles.
5. Model fuel economy regulations over time.

## 28 · Production Considerations

- Deploy for vehicle comparison tools.
- Use for fleet fuel cost forecasting.
- Monitor regulatory compliance (CAFE standards).
- Update with annual EPA test results.
- Output estimated annual fuel costs.

## 29 · Common Mistakes

1. Treating cylinders as continuous (it's step-wise).
2. Not creating power-to-weight ratio (the key physics metric).
3. Ignoring model year trend (technology improvements).
4. Treating AWD and FWD the same.
5. Not accounting for fuel type efficiency differences.

## 30 · Mini Challenge / Exercises

1. Plot MPG vs. 1/weight — is it more linear than MPG vs. weight?
2. Create `gal_per_100mi = 100/mpg` and retrain — does RMSE improve?
3. Remove fuel_type — how much does R² drop?
4. Plot MPG improvement trend by model_year.
5. Build separate models for Gasoline vs. Hybrid cars.

## 31 · Final Summary / Key Takeaways

1. **Weight** is the dominant MPG predictor (inverse relationship).
2. **Horsepower** reduces efficiency (power vs. economy tradeoff).
3. **Fuel type** (Hybrid > Diesel > Gasoline) sets efficiency tier.
4. **Model year** shows improving efficiency over time.
5. **Power-to-weight ratio** is the key engineered feature.
6. **Drivetrain** (AWD < RWD < FWD) affects efficiency measurably.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))